In [1]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

import os
import json
import pandas as pd
import traceback

from dotenv import load_dotenv

import PyPDF2

In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3
)

In [3]:
RESPONSE_JSON = {
    "1": {
        "mcq": "Multiple choice question",
        "options": {
            "a": "Option A",
            "b": "Option B",
            "c": "Option C",
            "d": "Option D"
        },
        "correct": "a"
    },
    "2": {
        "mcq": "Multiple choice question",
        "options": {
            "a": "Option A",
            "b": "Option B",
            "c": "Option C",
            "d": "Option D"
        },
        "correct": "b"
    },
    "3": {
        "mcq": "Multiple choice question",
        "options": {
            "a": "Option A",
            "b": "Option B",
            "c": "Option C",
            "d": "Option D"
        },
        "correct": "c"
    }
}

In [4]:
from langchain_core.prompts import PromptTemplate

quiz_generation_prompt = PromptTemplate(
    input_variables=[
        "text",
        "number",
        "subject",
        "tone",
        "response_json"
    ],
    template="""
You are an expert MCQ generator.

Your task is to create multiple-choice questions based ONLY on
the information provided in the text below.

Text:
{text}

Requirements:

- Generate exactly {number} MCQs.
- The questions are for {subject} students.
- Use a {tone} tone.
- Each question must have exactly four options.
- Use options a, b, c, and d.
- Clearly identify the correct answer.
- Do not repeat questions.
- Do not introduce information that is not present in the text.
- Make the questions meaningful and relevant to the provided text.
- Return the result in the JSON format shown below.

Expected JSON format:

{response_json}

Return ONLY the JSON object.
"""
)

In [5]:
quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject", "quiz"],
    template="""
You are an expert English grammarian, educator, and question-set evaluator.

You are given a multiple-choice quiz created for {subject} students.

Evaluate the quiz based on:

1. Clarity of the questions
2. Grammar and language quality
3. Relevance to the subject
4. Difficulty level
5. Quality of the options
6. Whether the correct answers are appropriate
7. Whether the questions test understanding rather than simple guessing

Provide a concise analysis using a maximum of 50 words.

If any question is unclear, grammatically incorrect, too easy, too difficult,
or inappropriate for the target students, identify it and suggest an improved version.

Quiz:

{quiz}

Provide your evaluation and suggested improvements.
"""
)

In [6]:
quiz_chain = quiz_generation_prompt | llm

review_chain = quiz_evaluation_prompt | llm

In [7]:
from langchain_core.runnables import RunnableLambda

In [9]:
def generate_quiz(inputs):
    return quiz_chain.invoke(inputs)

def generate_review(inputs):
    quiz = generate_quiz(inputs)

    review_input = {
        "subject": inputs["subject"],
        "quiz": quiz.content
    }

    review = review_chain.invoke(review_input)

    return {
        "quiz": quiz.content,
        "review": review.content
    }

generate_evaluate_chain = RunnableLambda(generate_review)

In [13]:
from pathlib import Path

file_path = Path("data/topic.txt")

with open(file_path, "r", encoding="utf-8") as file:
    TEXT = file.read()

print(TEXT)

Artificial Neural Networks

Artificial Neural Networks (ANNs) are computational models inspired by the structure and functioning of biological neural networks. They consist of interconnected artificial neurons organized into layers.

An ANN generally contains an input layer, one or more hidden layers, and an output layer. The input layer receives the features of the data. Hidden layers perform transformations using weights, biases, and activation functions. The output layer produces the final prediction.

During training, the network adjusts its weights and biases to reduce the difference between predicted and actual outputs. This process is commonly performed using backpropagation and gradient descent.

Activation functions such as ReLU, sigmoid, and tanh introduce non-linearity into the network, allowing it to learn complex relationships.

Artificial Neural Networks are used in image recognition, natural language processing, speech recognition, recommendation systems, and many other 

In [14]:
import json
import pandas as pd

In [15]:
# MCQ Configuration

NUMBER = 5
SUBJECT = "Data Science"
TONE = "simple"

In [16]:
# Invoke Generation + Evaluation
# ---------------------------------

response = generate_evaluate_chain.invoke(
    {
        "text": TEXT,
        "number": NUMBER,
        "subject": SUBJECT,
        "tone": TONE,
        "response_json": json.dumps(RESPONSE_JSON)
    }
)

In [17]:
# Display Evaluation
# ---------------------------------

print("========== REVIEW ==========")
print(response["review"])

========== REVIEW ==========
The quiz is clear, grammatically correct, and relevant. However, question 1 is too easy. 

Suggested improvement for question 1: "What characteristic of Biological Neural Networks inspired the development of Artificial Neural Networks?" 

This revised question tests understanding rather than simple recall.


In [18]:
# Convert Quiz JSON String
# to Python Dictionary
# ---------------------------------

quiz_str = response["quiz"]

quiz_dict = json.loads(quiz_str)

In [19]:
# Convert Quiz into Table Data
# ---------------------------------

quiz_table_data = []

for key, value in quiz_dict.items():

    mcq = value["mcq"]

    options = " | ".join(
        f"{option}: {option_value}"
        for option, option_value in value["options"].items()
    )

    correct = value["correct"]

    quiz_table_data.append({
        "MCQ": mcq,
        "Choices": options,
        "Correct": correct
    })

In [20]:
# Create DataFrame
# ---------------------------------

df = pd.DataFrame(quiz_table_data)

df

,MCQ,Choices,Correct
0,What is the main inspiration for Artificial Ne...,a: Biological Neural Networks | b: Machine Lea...,a
1,What is the purpose of the input layer in an A...,a: To produce the final prediction | b: To per...,c
2,What is the name of the process commonly used ...,a: Forward Propagation | b: Backpropagation an...,b
3,What is the role of activation functions such ...,a: To introduce linearity into the network | b...,c
4,In which of the following applications are Art...,a: Only in image recognition | b: Only in natu...,d
